In [ ]:
!wget -q https://github.com/ai-dock/llama.cpp-cuda/releases/download/v0.4.0/llama.cpp-v0.4.0-cuda-12.8-amd64.tar.gz
!tar -xzf llama.cpp-v0.4.0-cuda-12.8-amd64.tar.gz
!find . -maxdepth 3 -type f -name "llama-cli" -o -name "llama-server"

In [ ]:
!curl -sSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc \
  | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null \
  && echo "deb https://ngrok-agent.s3.amazonaws.com bookworm main" \
  | sudo tee /etc/apt/sources.list.d/ngrok.list \
  && sudo apt update \
  && sudo apt install ngrok

In [ ]:
!ngrok config add-authtoken <API-KEY>

In [ ]:
!export LD_LIBRARY_PATH=$PWD/cuda-12.8:$LD_LIBRARY_PATH && \
./cuda-12.8/llama-server \
  -hf unsloth/Qwen3.5-35B-A3B-GGUF:UD-Q4_K_XL \
  -ngl auto \
  -sm tensor \
  --flash-attn on \
  --cache-type-k q8_0 \
  --cache-type-v q8_0 \
  --ctx-size 131072 \
  -b 1024 \
  -ub 512 \
  -np 1 \
  --reasoning-preserve \
  --port 8080

In [ ]:
import os
import subprocess
import time

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = (
    os.path.abspath("./cuda-12.8")
    + ":"
    + env.get("LD_LIBRARY_PATH", "")
)

cmd = [
    "./cuda-12.8/llama-server",
    "-hf", "unsloth/Qwen3.8-27B-GGUF:UD-Q4_K_M",
    "-ngl", "auto",
    "-sm", "tensor",
    "--flash-attn", "on",
    "--cache-type-k", "q8_0",
    "--cache-type-v", "q8_0",
    "--ctx-size", "131072",
    "-b", "1024",
    "-ub", "512",
    "-np", "1",
    "--reasoning-preserve",
    "--port", "8080",
]

log_path = "llama.log"

print("🚀 Iniciando llama-server...")
print("📦 Cargando modelo...")
print("🧠 Preparando CUDA...")
print("⚡ Configurando Flash Attention...")
print("💾 Configurando KV cache...")
print("📚 Preparando contexto de 65K...")
print("⏳ Esperando a que el servidor esté listo...\n")

# Abrimos el log en modo escritura
log = open(log_path, "w", buffering=1)

process = subprocess.Popen(
    cmd,
    env=env,
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

print(f"🔄 llama-server iniciado (PID: {process.pid})")
print("📖 Leyendo logs...\n")

# Mensajes "animados"
mensajes = [
    "⏳ Cargando...",
    "🔧 Inicializando servidor...",
    "🧠 Cargando pesos del modelo...",
    "⚡ Inicializando GPU...",
    "📦 Preparando memoria...",
    "🚀 Levantando servidor...",
]

mensaje_idx = 0
ultima_posicion = 0

while True:
    # Si el proceso murió antes de estar listo
    if process.poll() is not None:
        log.close()

        print("\n❌ llama-server terminó antes de estar listo.")
        print(f"📄 Revisa el log: {log_path}")

        with open(log_path, "r", errors="replace") as f:
            print("\n--- Últimos logs ---")
            print(f.read()[-3000:])

        raise SystemExit(1)

    # Leer lo nuevo que haya aparecido en el log
    try:
        with open(log_path, "r", errors="replace") as f:
            f.seek(ultima_posicion)
            nuevos_logs = f.read()
            ultima_posicion = f.tell()

        if nuevos_logs:
            # Buscar el mensaje de servidor listo
            if "listening on http://127.0.0.1:8080" in nuevos_logs:
                print("\n" + "=" * 60)
                print("🟢 LISTO")
                print("🌐 llama-server está escuchando en:")
                print("   http://127.0.0.1:8080")
                print(f"🆔 PID: {process.pid}")
                print("📚 Contexto: 65,536 tokens")
                print("=" * 60)

                # Cerramos nuestro handle del log.
                # NO matamos llama-server: queda en segundo plano.
                log.close()

                print("\n✅ El proceso queda corriendo en segundo plano.")
                break

    except FileNotFoundError:
        pass

    # Mostrar progreso cada ~2 segundos
    print(mensajes[mensaje_idx], end="\r", flush=True)
    mensaje_idx = (mensaje_idx + 1) % len(mensajes)

    time.sleep(2)

In [ ]:
!pkill -9 -f llama-server

In [ ]:
import subprocess

subprocess.Popen(
    ["ngrok", "http", "8080"],
    stdout=open("ngrok.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True
)

print("🚀 ngrok corriendo en segundo plano")

In [ ]:
!pkill -9 -f ngrok

In [ ]:
!ngrok http 8080

In [ ]:
!cat llama.log

In [ ]:
!rm llama.log